# FATFORMER-XLA: PIPELINE HUẤN LUYỆN FINETUNE PHƯƠNG ÁN A
### MÁY CHỦ KAGGLE SERVER (GPU NVIDIA RTX PRO 6000 BLACKWELL 95GB VRAM)

> **Chế độ thực thi**: Hỗ trợ 100% chế độ **Save Version -> Save & Run All (Commit)** chạy ngầm chống đứt phiên.
> **Dữ liệu**: 144k ProGAN (85%) + 3.6k Staging GenImage (15%), Batch Size = 64.
> **Hàm mục tiêu**: DualStreamCELoss (label_smoothing=0.1) | Mỏ neo clean_prob = 0.50 xuyên suốt.

In [1]:
import os
import shutil
import glob
import zipfile

# 1. Tạo thư mục làm việc sạch
%cd /kaggle/working
!rm -rf fatformer-xla
os.makedirs("fatformer-xla", exist_ok=True)

# 2. In danh sách input hiện có để kiểm tra
print("=== DANH SÁCH CÁC DATASET ĐÃ ADD VÀO NOTEBOOK ===")
for item in os.listdir("/kaggle/input"):
    print(f" • /kaggle/input/{item}")

# 3. Tự động phát hiện vị trí mã nguồn (Kể cả khi Kaggle tự giải nén sẵn)
code_files = glob.glob("/kaggle/input/**/train.py", recursive=True)
zip_files = [z for z in glob.glob("/kaggle/input/**/*.zip", recursive=True) if "fatformer" in z.lower() or "code" in z.lower()]

if code_files:
    # Trường hợp A: Kaggle đã tự động giải nén sẵn các file
    src_dir = os.path.dirname(code_files[0])
    print(f"\n[✓] Phát hiện mã nguồn đã được giải nén sẵn tại: {src_dir}")
    shutil.copytree(src_dir, "/kaggle/working/fatformer-xla", dirs_exist_ok=True)

elif zip_files:
    # Trường hợp B: File zip còn nguyên
    target_zip = zip_files[0]
    print(f"\n[✓] Phát hiện file zip tại: {target_zip}")
    with zipfile.ZipFile(target_zip, 'r') as zip_ref:
        zip_ref.extractall("/kaggle/working/fatformer-xla")

else:
    print("\n[!] Chưa thấy mã nguồn! Vui lòng kiểm tra xem bạn đã bấm 'Add Input' dataset 'fatformer-code' chưa.")

# 4. Kiểm tra kết quả
%cd /kaggle/working/fatformer-xla
print("\n[✓] HOÀN TẤT THIẾT LẬP MÃ NGUỒN! Danh sách tệp trong fatformer-xla:")
!ls -lh


/kaggle/working
=== DANH SÁCH CÁC DATASET ĐÃ ADD VÀO NOTEBOOK ===
 • /kaggle/input/datasets
 • /kaggle/input/competitions

[✓] Phát hiện mã nguồn đã được giải nén sẵn tại: /kaggle/input/datasets/hoangleev/fatformer-code
/kaggle/working/fatformer-xla

[✓] HOÀN TẤT THIẾT LẬP MÃ NGUỒN! Danh sách tệp trong fatformer-xla:
total 36K
drwxr-xr-x 2 root root 4.0K Oct  8 20:44 configs
drwxr-xr-x 6 root root 4.0K Oct  8 20:44 src
drwxr-xr-x 2 root root 4.0K Oct  8 20:44 tools
-rw-r--r-- 1 root root  23K Oct  8 20:44 train.py


In [4]:
import os
import tarfile
import time

dest_data = "/kaggle/working/dataset_local"
os.makedirs(dest_data, exist_ok=True)
input_assets = "/kaggle/input/fatformer-assets-pack"

def extract_tar(tar_name, target_dir):
    # Tìm file tar trong input_assets (kể cả trong thư mục con)
    tar_candidates = [
        os.path.join(input_assets, tar_name),
        os.path.join(input_assets, "fatformer-assets-pack", tar_name)
    ]
    tar_path = None
    for c in tar_candidates:
        if os.path.exists(c):
            tar_path = c
            break
            
    if tar_path:
        print(f"[*] Đang giải nén {tar_name}...")
        t0 = time.time()
        with tarfile.open(tar_path, "r") as tar:
            tar.extractall(path=target_dir)
        print(f"[✓] Giải nén {tar_name} hoàn tất trong {time.time() - t0:.1f}s!")
    else:
        print(f"[!] Không tìm thấy {tar_name} tại {input_assets}")

# 1. Giải nén Pretrained Weights
extract_tar("pretrained.tar", "/kaggle/working/fatformer-xla")

# 2. Giải nén Dữ liệu huấn luyện & kiểm định
extract_tar("diffusion_staging.tar", dest_data)
extract_tar("progan_val.tar", dest_data)
extract_tar("progan_train.tar", dest_data)

print("\n[✓] Dữ liệu và trọng số mô hình đã giải nén xong!")
    

[!] Không tìm thấy pretrained.tar tại /kaggle/input/fatformer-assets-pack
[!] Không tìm thấy diffusion_staging.tar tại /kaggle/input/fatformer-assets-pack
[!] Không tìm thấy progan_val.tar tại /kaggle/input/fatformer-assets-pack
[!] Không tìm thấy progan_train.tar tại /kaggle/input/fatformer-assets-pack

[✓] Dữ liệu và trọng số mô hình đã giải nén xong!


In [5]:
import os
import glob
import shutil
import subprocess

clip_dir = "/kaggle/working/fatformer-xla/src/models/clip"
os.makedirs(clip_dir, exist_ok=True)
target_gz = os.path.join(clip_dir, "bpe_simple_vocab_16e6.txt.gz")

# 1. Tìm kiếm file BPE trên toàn bộ các thư mục Kaggle
bpe_gz_files = glob.glob("/kaggle/**/bpe_simple_vocab_16e6.txt.gz", recursive=True)
bpe_txt_files = glob.glob("/kaggle/**/bpe_simple_vocab_16e6.txt", recursive=True)

if bpe_gz_files:
    shutil.copy2(bpe_gz_files[0], target_gz)
    print(f"[✓] Đã copy file .gz từ: {bpe_gz_files[0]}")

elif bpe_txt_files:
    txt_path = bpe_txt_files[0]
    print(f"[*] Phát hiện file BPE dạng .txt (Kaggle đã tự giải nén) tại: {txt_path}")
    # Nén lại thành file .gz chuẩn theo yêu cầu của CLIP
    subprocess.run(f"gzip -c '{txt_path}' > '{target_gz}'", shell=True, check=True)
    print(f"[✓] Đã tái tạo thành công file: {target_gz}")

else:
    print("[!] Đang quét danh sách file BPE trên Kaggle:")
    !find /kaggle/ -name "*bpe*"

# Kiểm tra file đã sẵn sàng
if os.path.exists(target_gz):
    print(f"[✓] SẴN SÀNG! Kích thước file BPE: {os.path.getsize(target_gz):,} bytes")


[*] Phát hiện file BPE dạng .txt (Kaggle đã tự giải nén) tại: /kaggle/input/datasets/hoangleev/fatformer-code/src/models/clip/bpe_simple_vocab_16e6.txt
[✓] Đã tái tạo thành công file: /kaggle/working/fatformer-xla/src/models/clip/bpe_simple_vocab_16e6.txt.gz
[✓] SẴN SÀNG! Kích thước file BPE: 1,355,337 bytes


In [9]:
train_py_path = "/kaggle/working/fatformer-xla/train.py"

with open(train_py_path, "r", encoding="utf-8") as f:
    content = f.read()

if "from typing import Optional" not in content:
    content = "from typing import Optional, List, Dict, Any, Tuple\n" + content
    with open(train_py_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("[✓] Đã sửa xong file train.py trực tiếp trên Kaggle!")
else:
    print("[✓] File train.py đã có sẵn import typing.")


[✓] File train.py đã có sẵn import typing.


In [10]:
import os
%cd /kaggle/working/fatformer-xla

# 1. Tạo liên kết thư mục pretrained để nạp cả ViT-L-14.pt và checkpoint gốc CVPR 2024
!rm -rf pretrained
!ln -s /kaggle/input/datasets/hoangleev/fatformer-assets-pack/pretrained pretrained

# 2. Huấn Luyện Thật trên 144k ProGAN + 3.6k Diffusion + 8k Val
!python train.py \
    --progan_dir "/kaggle/input/datasets/hoangleev/fatformer-assets-pack/progan_train" \
    --staging_dir "/kaggle/input/datasets/hoangleev/fatformer-assets-pack/diffusion_staging" \
    --val_dir "/kaggle/input/datasets/hoangleev/fatformer-assets-pack/progan_val" \
    --clip_path "/kaggle/input/datasets/hoangleev/fatformer-assets-pack/pretrained/ViT-L-14.pt" \
    --batch_size 64 \
    --grad_accum 1 \
    --epochs 5 \
    --progan_ratio 0.85 \
    --staging_ratio 0.15 \
    --clean_prob 0.50 \
    --loss_type dual_stream_ce \
    --label_smoothing 0.1 \
    --lr 1e-4 \
    --output_dir "/kaggle/working/checkpoints/finetune_plan_a" \
    --checkpoint_name "fatformer_plan_a_final.pth"


/kaggle/working/fatformer-xla
FATFORMER-XLA: KHỞI ĐỘNG HỆ THỐNG HUẤN LUYỆN (MILESTONE 3 - TUẦN 4)
  • Thiết bị phần cứng:  cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)
  • Bộ nhớ VRAM:         94.97 GB VRAM
  • Batch Size thực:     64 (Tích lũy 1 bước -> Effective Batch Size: 64)
  • Số Epoch dự kiến:    5 (Bắt đầu từ Epoch 1)
  • Chế độ SRM:          True | Gating λ(x): True | Curriculum: True
  • Hàm mục tiêu Loss:   dual_stream_ce
  • Thư mục lưu Output:  /kaggle/working/checkpoints/finetune_plan_a
-------------------------------------------------------------------------------------
[DATASET] Kích hoạt Balanced DataLoader:
  • ProGAN: 144,024 ảnh (tỷ lệ mục tiêu: 85.0%)
  • Staging: 3,600 ảnh (tỷ lệ mục tiêu: 15.0%)
  • Tổng mẫu khả dụng: 147,624 ảnh | Batch: 64
[DATASET] Khởi tạo Validation DataLoader thành công: 8,000 ảnh từ /kaggle/input/datasets/hoangleev/fatformer-assets-pack/progan_val

[*] Đang khởi tạo mô hình FatFormer-XLA...
  [✓] Khởi tạo kiến trúc CLIP ViT-L/14 th

In [11]:
import os
import pandas as pd

history_csv = "/kaggle/working/checkpoints/finetune_plan_a/train_val_history.csv"
if os.path.exists(history_csv):
    df = pd.read_csv(history_csv)
    print("=== LỊCH SỬ HUẤN LUYỆN VÀ VALIDATION QUA CÁC EPOCH ===")
    display(df)
else:
    print("Chưa có file nhật ký lịch sử.")


=== LỊCH SỬ HUẤN LUYỆN VÀ VALIDATION QUA CÁC EPOCH ===


,epoch,train_loss,val_acc,val_ap,val_auc,val_r_acc,val_f_acc,epoch_time_s
0,1,0.555561,99.67,99.97,99.95,100.00,99.35,1947.38
1,2,0.451662,99.83,100.00,100.00,99.95,99.70,1945.57
2,3,0.455155,99.84,100.00,100.00,99.92,99.75,1921.38
3,4,0.450965,99.78,99.99,99.99,99.85,99.70,1955.20


In [12]:
import os

ckpt_dir = "/kaggle/working/checkpoints/finetune_plan_a"

# 1. Xóa file checkpoint epoch 4 bị lỗi do đầy đĩa và các epoch cũ thừa
for f in ["checkpoint_epoch_004.pth", "checkpoint_epoch_001.pth", "checkpoint_epoch_002.pth", "checkpoint_latest.pth"]:
    p = os.path.join(ckpt_dir, f)
    if os.path.exists(p):
        os.remove(p)
        print(f"[✓] Đã xóa {f} để giải phóng đĩa.")

print("\n--- DANH SÁCH CHECKPOINT CÒN LẠI HOÀN CHỈNH ---")
for f in sorted(os.listdir(ckpt_dir)):
    p = os.path.join(ckpt_dir, f)
    sz = os.path.getsize(p) / (1024**3)
    print(f" • {f}: {sz:.2f} GB")


[✓] Đã xóa checkpoint_epoch_004.pth để giải phóng đĩa.
[✓] Đã xóa checkpoint_epoch_001.pth để giải phóng đĩa.
[✓] Đã xóa checkpoint_epoch_002.pth để giải phóng đĩa.
[✓] Đã xóa checkpoint_latest.pth để giải phóng đĩa.

--- DANH SÁCH CHECKPOINT CÒN LẠI HOÀN CHỈNH ---
 • checkpoint_epoch_003.pth: 3.28 GB
 • model_best.pth: 3.28 GB
 • train_val_history.csv: 0.00 GB


In [13]:
import os
import shutil

ckpt_dir = "/kaggle/working/checkpoints/finetune_plan_a"
best_model = os.path.join(ckpt_dir, "model_best.pth")
final_model = os.path.join(ckpt_dir, "fatformer_plan_a_final.pth")

# Tạo bản sao chính thức cho Báo Cáo Đồ Án
if os.path.exists(best_model) and not os.path.exists(final_model):
    shutil.copyfile(best_model, final_model)
    print(f"[✓] Đã tạo checkpoint chính thức: {final_model}")

print("\n=== TỔNG KẾT TÀI SẢN HOÀN THÀNH HUẤN LUYỆN ===")
for f in sorted(os.listdir(ckpt_dir)):
    p = os.path.join(ckpt_dir, f)
    print(f" • {f}: {os.path.getsize(p)/(1024**3):.2f} GB")


[✓] Đã tạo checkpoint chính thức: /kaggle/working/checkpoints/finetune_plan_a/fatformer_plan_a_final.pth

=== TỔNG KẾT TÀI SẢN HOÀN THÀNH HUẤN LUYỆN ===
 • checkpoint_epoch_003.pth: 3.28 GB
 • fatformer_plan_a_final.pth: 3.28 GB
 • model_best.pth: 3.28 GB
 • train_val_history.csv: 0.00 GB


In [18]:
import os
from IPython.display import FileLink

# 1. Đảm bảo đứng ở thư mục gốc /kaggle/working
%cd /kaggle/working

# 2. Đường dẫn file to 3.28 GB chính thức của Epoch 3
file_3gb = "checkpoints/finetune_plan_a/fatformer_plan_a_final.pth"

if os.path.exists(file_3gb):
    sz_gb = os.path.getsize(file_3gb) / (1024**3)
    print(f"[✓] ĐÃ TÌM THẤY FILE TO CHÍNH THỨC ({sz_gb:.2f} GB)!")
    print("\n👉 BẤM VÀO ĐƯỜNG LINK MÀU XANH DƯỚI ĐÂY ĐỂ TẢI VỀ MÁY TÍNH:")
    display(FileLink(file_3gb))
else:
    print(f"[!] Không tìm thấy file tại: {file_3gb}")


/kaggle/working
[✓] ĐÃ TÌM THẤY FILE TO CHÍNH THỨC (3.28 GB)!

👉 BẤM VÀO ĐƯỜNG LINK MÀU XANH DƯỚI ĐÂY ĐỂ TẢI VỀ MÁY TÍNH:


/kaggle/working/checkpoints/finetune_plan_a/fatformer_plan_a_final.pth

In [19]:
import os
from IPython.display import FileLink

csv_path = "checkpoints/finetune_plan_a/train_val_history.csv"
if os.path.exists(csv_path):
    print("[✓] Link tải file lịch sử số liệu huấn luyện:")
    display(FileLink(csv_path))
else:
    print(f"[!] Không tìm thấy {csv_path}, kiểm tra đường dẫn thư mục hiện tại.")


[✓] Link tải file lịch sử số liệu huấn luyện:


/kaggle/working/checkpoints/finetune_plan_a/train_val_history.csv

In [22]:
!rm -rf /kaggle/working/dataset_local
!rm -rf /kaggle/working/checkpoints/finetune_plan_a/checkpoint_epoch_003.pth
